In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df = pd.read_csv("/Users/serhatguldogan/Library/CloudStorage/OneDrive-KocUniversitesi/Project_/RAW DATA/BTC_1sec.csv")


We do the exact same things that we did at "4_states_transitions_and_means.ipynb". Refer to that.

In [7]:
# Calculate price changes
df['price_change'] = df['midpoint'].diff()

# Function to discretize price changes
def discretize_price_change(p):
    """
    Discretize price changes:
    - 0 < p ≤ 5 → 5
    - 5 < p ≤ 10 → 10
    - 10 < p ≤ 15 → 15
    - 15 < p ≤ 20 → 20
    Same for negatives (symmetric)
    - 0 stays 0
    """
    if pd.isna(p) or p == 0:
        return 0
    elif p > 0:
        if 0 < p <= 5.0:
            return 5.0
        elif 5.0 < p <= 10:
            return 10
        elif 10 < p <= 15:
            return 15
        
        else:  # p > 87.5
            return 20
    else:  # p < 0
        if -5 <= p < 0:
            return -5
        elif -10.0 <= p < -5.0:
            return -10.0
        elif -15.0 <= p < -10.0:
            return -15.0
        
        else:  
            return -20

# Apply discretization
df['price_change_discretized'] = df['price_change'].apply(discretize_price_change)

# Create label column: map discretized values to labels
# +5.0 → +1, +10 → +2, +15 → +3, +20 → +4 
# -5.0 → -1, -10.0 → -2, -15.0 → -3, -20.0 → -4
# 0 → 0
def label_price_change(discretized_value):
    if discretized_value == 0:
        return 0
    elif discretized_value > 0:
        return int(discretized_value / 5)
    else:  # discretized_value < 0
        return int(discretized_value / 5)

df['price_change_label'] = df['price_change_discretized'].apply(label_price_change)

# Display results
print("First 20 rows with price changes:")
print(df[['midpoint', 'price_change', 'price_change_discretized', 'price_change_label']].head(20))
print("\nPrice change label distribution:")
print(df['price_change_label'].value_counts().sort_index())


First 20 rows with price changes:
     midpoint  price_change  price_change_discretized  price_change_label
0   56035.995           NaN                       0.0                   0
1   56035.995         0.000                       0.0                   0
2   56035.995         0.000                       0.0                   0
3   56035.995         0.000                       0.0                   0
4   56035.995         0.000                       0.0                   0
5   56035.995         0.000                       0.0                   0
6   56035.995         0.000                       0.0                   0
7   56035.995         0.000                       0.0                   0
8   56035.995         0.000                       0.0                   0
9   56035.995         0.000                       0.0                   0
10  56035.995         0.000                       0.0                   0
11  56035.995         0.000                       0.0                   0
12  

In [8]:
changes = df["price_change_label"]

In [9]:
from collections import Counter

# Count the number of transitions between states in 'changes'
transition_counter = Counter()
for i in range(1, len(changes)):
    prev_state = changes.iloc[i - 1]
    curr_state = changes.iloc[i]
    
    transition_counter[(prev_state, curr_state)] += 1


In [10]:
counter = dict(transition_counter)
counter

{(0, 0): 491297,
 (0, -3): 6105,
 (-3, -2): 1713,
 (-2, 2): 2340,
 (2, -1): 6315,
 (-1, -4): 3376,
 (-4, -4): 2266,
 (-4, 1): 2546,
 (1, -2): 4696,
 (-2, -2): 3387,
 (-2, 1): 6933,
 (1, 1): 42669,
 (1, -4): 1283,
 (1, 0): 45331,
 (0, 3): 6004,
 (3, -4): 392,
 (-4, -3): 1237,
 (-3, -1): 3458,
 (-1, 4): 1048,
 (4, 4): 2186,
 (4, 1): 3691,
 (1, 2): 8466,
 (-1, -3): 3074,
 (-3, 4): 427,
 (4, 3): 1296,
 (3, 1): 3831,
 (-4, 0): 3997,
 (0, 4): 5762,
 (4, -4): 1050,
 (-4, 2): 1188,
 (-1, 1): 37576,
 (1, -1): 37031,
 (-3, -4): 1164,
 (4, -2): 950,
 (1, -3): 1317,
 (-4, 3): 594,
 (3, -3): 382,
 (-1, -1): 34590,
 (2, -2): 2191,
 (-2, -4): 1954,
 (1, 4): 3171,
 (4, -3): 475,
 (-3, 3): 380,
 (3, 0): 4435,
 (3, -1): 2005,
 (-1, 2): 4532,
 (2, 1): 9853,
 (2, 3): 1721,
 (-4, 4): 1149,
 (0, 1): 37733,
 (-4, -2): 2018,
 (-2, -1): 8204,
 (1, 3): 3203,
 (4, 2): 2187,
 (2, 0): 11881,
 (2, 4): 1824,
 (-3, 1): 2335,
 (4, 0): 3414,
 (3, 2): 1995,
 (2, 2): 3859,
 (-3, 0): 4399,
 (0, -2): 13265,
 (-2, 0): 10584

In [11]:
d=counter.copy()
for key in counter.keys():
    if 0 in key:
        d.pop(key)

In [12]:
d

{(-3, -2): 1713,
 (-2, 2): 2340,
 (2, -1): 6315,
 (-1, -4): 3376,
 (-4, -4): 2266,
 (-4, 1): 2546,
 (1, -2): 4696,
 (-2, -2): 3387,
 (-2, 1): 6933,
 (1, 1): 42669,
 (1, -4): 1283,
 (3, -4): 392,
 (-4, -3): 1237,
 (-3, -1): 3458,
 (-1, 4): 1048,
 (4, 4): 2186,
 (4, 1): 3691,
 (1, 2): 8466,
 (-1, -3): 3074,
 (-3, 4): 427,
 (4, 3): 1296,
 (3, 1): 3831,
 (4, -4): 1050,
 (-4, 2): 1188,
 (-1, 1): 37576,
 (1, -1): 37031,
 (-3, -4): 1164,
 (4, -2): 950,
 (1, -3): 1317,
 (-4, 3): 594,
 (3, -3): 382,
 (-1, -1): 34590,
 (2, -2): 2191,
 (-2, -4): 1954,
 (1, 4): 3171,
 (4, -3): 475,
 (-3, 3): 380,
 (3, -1): 2005,
 (-1, 2): 4532,
 (2, 1): 9853,
 (2, 3): 1721,
 (-4, 4): 1149,
 (-4, -2): 2018,
 (-2, -1): 8204,
 (1, 3): 3203,
 (4, 2): 2187,
 (2, 4): 1824,
 (-3, 1): 2335,
 (3, 2): 1995,
 (2, 2): 3859,
 (-2, -3): 1647,
 (-3, 2): 925,
 (2, -4): 612,
 (-1, 3): 1169,
 (3, -2): 784,
 (-4, -1): 3738,
 (-3, -3): 952,
 (4, -1): 1974,
 (-2, 4): 562,
 (3, 3): 1055,
 (3, 4): 1094,
 (2, -3): 564,
 (-2, 3): 551,
 (-

In [13]:
sum_1x = sum(v for (a, b), v in d.items() if a == 1)
normalized_1y = {(a, b): v / sum_1x for (a, b), v in d.items() if a == 1}
normalized_1y


{(1, -2): 0.04611335873365018,
 (1, 1): 0.4189972112023253,
 (1, -4): 0.012598688086727679,
 (1, 2): 0.0831336658941828,
 (1, -1): 0.36363368553360303,
 (1, -3): 0.012932558230881025,
 (1, 4): 0.031138300797360463,
 (1, 3): 0.03145253152126949}

In [14]:
normalized_by_first = {}
for a in set(key[0] for key in d.keys()):
    sum_ax = sum(v for (x, y), v in d.items() if x == a)
    if sum_ax == 0:
        continue
    for (x, y), v in d.items():
        if x == a:
            normalized_by_first[(x, y)] = v / sum_ax
normalized_by_first


{(1, -2): 0.04611335873365018,
 (1, 1): 0.4189972112023253,
 (1, -4): 0.012598688086727679,
 (1, 2): 0.0831336658941828,
 (1, -1): 0.36363368553360303,
 (1, -3): 0.012932558230881025,
 (1, 4): 0.031138300797360463,
 (1, 3): 0.03145253152126949,
 (2, -1): 0.23441850105794573,
 (2, -2): 0.08133189799175916,
 (2, 1): 0.3657522550948439,
 (2, 3): 0.06388507368499202,
 (2, 4): 0.06770852667136865,
 (2, 2): 0.14324956382939233,
 (2, -4): 0.02271799250157764,
 (2, -3): 0.020936189168120567,
 (3, -4): 0.03397469232102617,
 (3, 1): 0.33203328133125326,
 (3, -3): 0.03310799098630612,
 (3, -1): 0.1737736176113711,
 (3, 2): 0.17290691627665106,
 (3, -2): 0.06794938464205234,
 (3, 3): 0.09143699081296586,
 (3, 4): 0.09481712601837407,
 (4, 4): 0.1583025563038598,
 (4, 1): 0.26728944891013107,
 (4, 3): 0.09385183575928742,
 (4, -4): 0.07603736693460786,
 (4, -2): 0.06879571294083568,
 (4, -3): 0.03439785647041784,
 (4, 2): 0.15837497284379753,
 (4, -1): 0.14295024983706278,
 (-1, -4): 0.036488224549

In [15]:
t=0
for key in normalized_by_first.keys():
    if key[0]==4:
        t+=normalized_by_first[key]

t

1.0

In [16]:
import numpy as np

# matrix will be ordered so rows/cols go from -4 to 4 skipping 0, i.e.
# indices: [-4, -3, -2, -1, 1, 2, 3, 4] but as 0 is omitted (like the dict d)
# But requested: 6x6, so use [-4, -3, -2, -1, 1, 2]

indices = [-4, -3, -2, -1, 1, 2,3,4]
size = len(indices)
mat = np.zeros((size, size))

# fill mat[i,j] with normalized_by_first[(row_val,col_val)]
for i, row in enumerate(indices):
    for j, col in enumerate(indices):
        key = (row, col)
        if key in normalized_by_first:
            mat[i, j] = normalized_by_first[key]
        else:
            mat[i, j] = 0.0  # or np.nan if preferred

mat  # this will display the matrix in the notebook


array([[0.15377307, 0.08394408, 0.13694354, 0.2536645 , 0.17277416,
        0.08061889, 0.04030945, 0.07797231],
       [0.10251894, 0.0838471 , 0.15087194, 0.30456227, 0.20565439,
        0.08146909, 0.03346838, 0.03760789],
       [0.07639378, 0.06439127, 0.13241848, 0.32074439, 0.27105325,
        0.09148487, 0.02154195, 0.02197201],
       [0.03648822, 0.03322417, 0.07736455, 0.37385299, 0.40612604,
        0.04898242, 0.0126347 , 0.01132691],
       [0.01259869, 0.01293256, 0.04611336, 0.36363369, 0.41899721,
        0.08313367, 0.03145253, 0.0311383 ],
       [0.02271799, 0.02093619, 0.0813319 , 0.2344185 , 0.36575226,
        0.14324956, 0.06388507, 0.06770853],
       [0.03397469, 0.03310799, 0.06794938, 0.17377362, 0.33203328,
        0.17290692, 0.09143699, 0.09481713],
       [0.07603737, 0.03439786, 0.06879571, 0.14295025, 0.26728945,
        0.15837497, 0.09385184, 0.15830256]])

In [17]:
np.savetxt("mat_8_states.csv",X=mat,fmt="%f",delimiter=",")

In [18]:

# Solve the equation vP = v where P is the matrix "mat" (i.e., find the stationary distribution)

# To solve vP = v, equivalently solve v(P - I) = 0 with constraint sum(v) = 1
# We transpose so that v is a row vector: v @ P = v <=> (P.T - I.T) v^T = 0
from numpy.linalg import svd

P = mat
size = P.shape[0]
A = P.T - np.eye(size)  # (P^T - I) so that nullspace is stationary v^T

# Append the normalization equation sum(v) = 1
A = np.vstack([A, np.ones(size)])
b = np.zeros(size + 1)
b[-1] = 1

# Solve using least squares (the last row enforces sum(v) = 1)
v, residuals, rank, s = np.linalg.lstsq(A, b, rcond=None)

# v is the stationary distribution
v = v.real  # ensure no tiny imaginary part from solver

v = np.array(v) # Display the stationary distribution vector


This is the stationary probability distribution vector.

In [19]:
np.savetxt(fname="pi_8_states.csv",X=v,
fmt = "%f",delimiter=",")

In [20]:
# Calculate mean price change from stationary distribution
# Each state j corresponds to price change = j * 5
bucket_size = 5.0

# Calculate mean: E[X] = Σ (state_j * price_change_j * probability_j)
# where price_change_j = state_j * bucket_size
mean_price_change = 0.0

print("State | Price Change | Probability | Contribution")
print("-" * 60)
for i, state in enumerate(indices):
    price_change = state * bucket_size
    prob = v[i]
    contribution = price_change * prob
    mean_price_change += contribution
    if abs(contribution) > 0.001:  # Only print significant contributions
        print(f"{state:5d} | {price_change:11.2f} | {prob:10.6f} | {contribution:12.6f}")

print("-" * 60)
print(f"\nMean price change (from stationary distribution): {mean_price_change:.6f}")


State | Price Change | Probability | Contribution
------------------------------------------------------------
   -4 |      -20.00 |   0.036979 |    -0.739573
   -3 |      -15.00 |   0.030335 |    -0.455032
   -2 |      -10.00 |   0.073850 |    -0.738503
   -1 |       -5.00 |   0.333538 |    -1.667692
    1 |        5.00 |   0.375693 |     1.878467
    2 |       10.00 |   0.082681 |     0.826814
    3 |       15.00 |   0.031614 |     0.474209
    4 |       20.00 |   0.035308 |     0.706168
------------------------------------------------------------

Mean price change (from stationary distribution): 0.284859
